# Запуск Windows (.EXE) в Google Colab и стриминг экрана на телефон 📱🖥️

Этот блокнот позволяет вам запускать любые Windows-приложения (.exe) на виртуальном сервере Google Colab и транслировать полноценный графический рабочий стол на ваш телефон, планшет или компьютер **абсолютно бесплатно и без регистрации**.

### Как это работает:
1. **Wine** запускает Windows-приложение (.exe) в среде Linux.
2. **Xvfb & Fluxbox** создают виртуальный экран высокого разрешения прямо в оперативной памяти Colab.
3. **x11vnc & noVNC** превращают этот экран в интерактивный веб-сайт (стрим).
4. **Cloudflare Tunnel** создает безопасный и быстрый публичный веб-адрес (туннель) и QR-код, по которым вы можете зайти с любого мобильного телефона через обычный браузер.

---
## 🛠️ Шаг 1: Установка системных утилит, Wine и графической оболочки
Нажмите на кнопку запуска ячейки ниже (круглая кнопка Play), чтобы установить Wine (для запуска `.exe`), виртуальный дисплей, VNC и noVNC. Это займет около 1–2 минут.

In [ ]:
#@title Нажмите Play для установки окружения { display-mode: "form" }

import os
from IPython.display import clear_output

print("🔄 1/5 Обновление пакетного менеджера...")
os.system("apt-get update -qq")

print("🔄 2/5 Установка системных и графических утилит (Xvfb, Fluxbox, x11vnc)...")
# Устанавливаем базовые графические пакеты отдельно, они всегда ставятся без проблем
os.system("apt-get install -y --no-install-recommends xvfb x11vnc fluxbox wget curl git python3-pip")

print("🔄 3/5 Включение поддержки 32-битной архитектуры...")
os.system("dpkg --add-architecture i386")
os.system("apt-get update -qq")

print("🔄 4/5 Установка Wine (основной пакет и 64-битная версия)...")
# Устанавливаем базовую и 64-битную версию Wine
os.system("apt-get install -y --no-install-recommends wine64 wine")

print("🔄 4.5/5 Попытка установки поддержки 32-битного Wine (если поддерживается)...")
# Установка 32-битной версии отдельно, чтобы ее возможные конфликты не ломали основную установку
os.system("apt-get install -y --no-install-recommends wine32")

print("🔄 5/5 Установка noVNC и websockify для трансляции в браузер...")
if not os.path.exists('/opt/noVNC'):
    os.system("git clone --depth 1 https://github.com/novnc/noVNC.git /opt/noVNC")
if not os.path.exists('/opt/noVNC/utils/websockify'):
    os.system("git clone --depth 1 https://github.com/novnc/websockify /opt/noVNC/utils/websockify")

clear_output()
print("✅ Шаг 1 успешно завершен! Все компоненты, включая Wine, готовы к работе.")

---
## 🖥️ Шаг 2: Запуск виртуального экрана и VNC-сервера
Запустите эту ячейку, чтобы запустить графическую оболочку в фоновом режиме.

In [ ]:
#@title Нажмите Play для запуска виртуального дисплея { display-mode: "form" }

import os
import subprocess
import time

print("🔄 Очистка старых процессов (если они были запущены ранее)...")
os.system("pkill -f Xvfb")
os.system("pkill -f fluxbox")
os.system("pkill -f x11vnc")
os.system("pkill -f novnc_proxy")
time.sleep(1)

print("🔄 Запуск виртуального дисплея (Xvfb) с разрешением 1280x720...")
subprocess.Popen("Xvfb :99 -screen 0 1280x720x24", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1.5)

print("🔄 Запуск менеджера окон (Fluxbox)...")
subprocess.Popen("DISPLAY=:99 fluxbox", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1.5)

print("🔄 Запуск VNC сервера...")
subprocess.Popen("DISPLAY=:99 x11vnc -forever -nopw -listen 127.0.0.1 -rfbport 5900 -shared", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1.5)

print("🔄 Запуск noVNC прокси-сервера...")
subprocess.Popen("/opt/noVNC/utils/novnc_proxy --vnc localhost:5900 --listen 6080", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2.5)

print("✅ Виртуальный дисплей и noVNC успешно работают в фоновом режиме!")

---
## 🌐 Шаг 3: Создание публичного туннеля и QR-кода
Запустите ячейку ниже. Она установит Cloudflare Tunnel, получит публичную ссылку и выведет удобный **QR-код**. Вы сможете отсканировать его камерой телефона, чтобы сразу открыть экран трансляции в браузере мобильного!

In [ ]:
#@title Нажмите Play для создания туннеля { display-mode: "form" }

import os
import subprocess
import re
import time
import urllib.parse
from IPython.display import display, HTML, clear_output

print("🔄 Запуск Cloudflare Tunnel...")

# Установка cloudflared при необходимости
if not os.path.exists('/usr/local/bin/cloudflared') and not os.path.exists('/usr/bin/cloudflared'):
    print("📥 Скачивание и установка утилиты cloudflared...")
    os.system("wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
    os.system("chmod +x /usr/local/bin/cloudflared")

# Очистка старых туннелей
os.system("pkill -f cloudflared")
time.sleep(1)

# Запуск нового туннеля в фоне
cmd = "cloudflared tunnel --url http://127.0.0.1:6080"
process = subprocess.Popen(cmd.split(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

url = None
print("🔍 Ожидание создания публичной безопасной ссылки...")
for i in range(40):
    line = process.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        # Оптимальные параметры для телефона: автоподключение и автомасштабирование экрана
        vnc_url = f"{url}/vnc.html?autoconnect=true&resize=scale"
        
        clear_output()
        
        # Создаем QR-код с помощью открытого API
        qr_api_url = f"https://api.qrserver.com/v1/create-qr-code/?size=250x250&data={urllib.parse.quote(vnc_url)}"
        
        html_code = f"""
        <div style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; padding: 25px; border: 3px dashed #4CAF50; border-radius: 15px; background-color: #fcfcfc; max-width: 550px; margin: 20px auto; text-align: center; box-shadow: 0 4px 15px rgba(0,0,0,0.1);">
            <h2 style="color: #4CAF50; margin-top: 0; font-size: 26px;">🎉 Стрим готов! 🎉</h2>
            <p style="font-size: 16px; margin: 10px 0; color: #333;">Откройте эту ссылку на вашем телефоне или компьютере:</p>
            <p style="font-weight: bold; font-size: 20px; margin: 20px 0;">
                <a href="{vnc_url}" target="_blank" style="color: #ffffff; background-color: #4CAF50; padding: 12px 25px; border-radius: 8px; text-decoration: none; display: inline-block; box-shadow: 0 3px 6px rgba(76,175,80,0.4);">👉 Открыть Трансляцию Экрана 👈</a>
            </p>
            <p style="color: #666; font-size: 15px;">Или просто отсканируйте этот QR-код камерой мобильного:</p>
            <div style="margin: 25px 0;">
                <img src="{qr_api_url}" alt="QR Code" style="border: 3px solid #eee; padding: 8px; background: white; border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.05);" />
            </div>
            <div style="text-align: left; background-color: #f1f8e9; padding: 15px; border-left: 5px solid #8bc34a; border-radius: 4px; font-size: 13px; color: #33691e; line-height: 1.5;">
                💡 <b>Советы по управлению на телефоне:</b><br>
                • Свайп пальцем слева направо открывает боковое меню VNC (там есть настройки, клавиатура и клики).<br>
                • В настройках (иконка шестерёнки) убедитесь, что включено <b>Local Scaling</b> (Локальное масштабирование), чтобы весь экран VNC помещался в экран вашего мобильного телефона.<br>
                • Используйте жест сведения/разведения пальцев для зума экрана при необходимости.
            </div>
        </div>
        """
        display(HTML(html_code))
        break
    time.sleep(0.1)

if not url:
    print("❌ Ошибка: не удалось получить ссылку туннеля. Пожалуйста, перезапустите эту ячейку.")

---
## 📥 Шаг 4: Скачивание вашего .EXE файла или архива
Вставьте ссылку на файл программы, которую хотите скачать на виртуальный сервер Colab. Укажите имя, под которым его сохранить (например, `installer.exe`).

In [ ]:
download_url = "https://example.com/file.exe" #@param {type:"string"}
output_filename = "my_program.exe" #@param {type:"string"}

if download_url and download_url != "https://example.com/file.exe":
    print(f"🔄 Скачивание файла: {output_filename}...")
    import os
    os.system(f'wget -O "{output_filename}" "{download_url}"')
    print(f"✅ Скачивание завершено! Файл сохранен как: {output_filename}")
else:
    print("ℹ️ Пожалуйста, вставьте СВОЮ прямую ссылку на .exe файл (или архив) выше и запустите ячейку вновь.")
    print("Для примера, вы можете загрузить любую известную вам программу через wget напрямую.")

---
## 🚀 Шаг 5: Запуск .EXE программы в виртуальном дисплее
Введите имя вашего `.exe` файла и нажмите кнопку воспроизведения. Программа откроется на вашем виртуальном дисплее, и вы сразу увидите её графический интерфейс на экране вашего телефона (через открытую ранее вкладку трансляции)!

In [ ]:
exe_to_run = "my_program.exe" #@param {type:"string"}

import os
import subprocess

if os.path.exists(exe_to_run):
    print(f"🚀 Запуск {exe_to_run} на дисплее :99 с помощью Wine...")
    # Запускаем приложение в фоне на дисплее :99, чтобы оно транслировалось в VNC
    subprocess.Popen(f"DISPLAY=:99 wine {exe_to_run}", shell=True)
    print("✅ Приложение запущено! Взгляните на вкладку трансляции на вашем телефоне.")
else:
    print(f"❌ Ошибка: Файл {exe_to_run} не найден. Сначала скачайте его на Шаге 4 или проверьте имя.")

---
## 🎮 Как играть в игры и использовать джойстик/геймпад на телефоне?

Если вы запустили игру или приложение, где требуется управление джойстиком или кнопками WASD/стрелками, у вас есть **три отличных способа** сделать это:

### Способ 1: Использование мобильного приложения-геймпада (Самый удобный)
На телефоне можно установить виртуальную клавиатуру, которая выглядит и работает как полноценный игровой геймпад. 
* **Для Android:** скачайте бесплатное приложение **GamePad** (от Fishstix) или **Hacker's Keyboard** из Google Play.
  1. Включите установленный GamePad в настройках языков ввода телефона.
  2. Настройте кнопки геймпада (например, D-Pad повесить на стрелочки клавиатуры или клавиши WASD, а кнопки A, B, X, Y на пробел, Enter и т.д.).
  3. Когда вы откроете стрим в браузере и нажмете на экран ввода текста, вместо обычной клавиатуры появится крутой игровой геймпад прямо поверх стрима!

### Способ 2: Подключение реального геймпада по Bluetooth 🎮
Если у вас есть физический геймпад (например, от **Xbox, PlayStation DualShock** или любой мобильный Bluetooth-геймпад):
1. Подключите геймпад к телефону через Bluetooth.
2. Откройте страницу трансляции в мобильном браузере.
3. Браузер автоматически распознает подключенный контроллер, и вы сможете использовать его для игр!

### Способ 3: Экранные кнопки в самом браузере
Вы также можете установить мобильные браузеры, поддерживающие кастомные оверлеи клавиш, либо использовать сторонние расширения для вывода стрелочек поверх сайтов.

---
## ⚠️ Советы и Решение проблем

### 1. Как запустить другую программу или скачать дополнительные файлы в процессе?
Вы можете в любое время скачивать дополнительные файлы (Шаг 4) и запускать новые `.exe` файлы (Шаг 5). Они все будут открываться на одном и том же рабочем столе и мгновенно стримиться на ваш телефон.

### 2. Приложение выдает ошибку о нехватке библиотек (например, .NET)?
Вы можете установить дополнительные компоненты Windows (такие как .NET, DirectX, шрифты) с помощью утилиты `winetricks`. Для этого создайте новую ячейку кода и выполните:
```bash
!winetricks dotnet48
```
или установите необходимые шрифты:
```bash
!winetricks corefonts
```

### 3. Курсор мыши не совпадает или неудобно кликать на телефоне?
В боковом меню noVNC (свайп слева направо) есть иконка с шестеренкой. Там вы можете изменить режим мыши. Попробуйте переключить режимы **Mouse Control** (например, с Direct на Pointer), чтобы найти максимально удобный для вашего телефона.

### 4. Как посмотреть, что происходит в процессах Colab?
Если вам кажется, что приложение зависло, вы можете посмотреть список запущенных процессов:
```bash
!ps aux | grep wine
```